In [40]:
#檢查ID是否重複

import pandas as pd
import arcpy

# 1. 設定參數
input_feature = "OL_P1_2204_coral_Simplify"
field_name = "ID"  # 確保這是正確的欄位名稱

# 2. 使用 Search Cursor 讀取資料 (比 NumPyArray 更能包容 Null 值)
data_list = []
with arcpy.da.SearchCursor(input_feature, [field_name]) as cursor:
    for row in cursor:
        # 如果是 None (Null)，轉為字串方便後續統計，否則保留原值
        val = row[0] if row[0] is not None else "<Null>"
        data_list.append(val)

# 3. 轉換為 DataFrame
df = pd.DataFrame(data_list, columns=[field_name])

# 4. 找出重複的行
# keep=False 表示標記所有重複出現的值
dup_df = df[df.duplicated(subset=[field_name], keep=False)]

# 5. 判斷並印出結果
print(f"--- 檢查報告: {field_name} ---")
if dup_df.empty:
    print(f"恭喜！欄位 '{field_name}' 中沒有發現任何重複值。")
else:
    # 統計重複次數
    result = dup_df[field_name].value_counts()
    print(f"注意：發現 {len(result)} 組重複數值！")
    print("-" * 30)
    print(result)
    print("-" * 30)

--- 檢查報告: ID ---
恭喜！欄位 'ID' 中沒有發現任何重複值。


In [41]:
#檢查Temp_ID在G S有無重複 在Fi Fu重複沒關係 但不能和G S有重疊
import arcpy
import pandas as pd

# 1. 基本設定
input_feature = "OL_P1_2204_coral_Simplify"
fate_field = "Fate"
id_field = "Temp_ID"
real_id_field = "ID"

# 定義群組描述 (這是您在 ArcGIS 介面看到的文字)
group_a_desc = ['G', 'S', 'GorS']
group_b_desc = ['Fi', 'Fu']

# --- 自動取得 Domain 代號對照 ---
def get_domain_map(feature, field_name):
    desc = arcpy.Describe(feature)
    workspace = desc.path
    domains = arcpy.da.ListDomains(workspace)
    field_domain_name = next((f.domain for f in desc.fields if f.name == field_name), "")
    
    if field_domain_name:
        for d in domains:
            if d.name == field_domain_name:
                return d.codedValues # 格式如 {1: "G", 2: "S"}
    return {}

# 取得 Domain 對照表
domain_dict = get_domain_map(input_feature, fate_field)
# 反向對照表：將 "G" 轉回 1
rev_domain_dict = {str(v).strip(): k for k, v in domain_dict.items()}

# 轉換搜尋代號 (確保匹配 Domain 中的原始型態)
group_a_codes = [rev_domain_dict.get(d, d) for d in group_a_desc]
group_b_codes = [rev_domain_dict.get(d, d) for d in group_b_desc]

def get_detailed_df(codes_list):
    """抓取詳細資料並轉換為 DataFrame"""
    data = []
    # 建立 SQL 語句
    formatted_codes = ",".join([f"'{c}'" if isinstance(c, str) else str(c) for c in codes_list])
    where = f"{fate_field} IN ({formatted_codes})"
    
    with arcpy.da.SearchCursor(input_feature, [id_field, real_id_field, fate_field], where_clause=where) as cursor:
        for row in cursor:
            # 統一將 Temp_ID 轉為字串避免排序錯誤
            t_id = str(row[0]) if row[0] is not None else "<Null>"
            r_id = str(row[1]) if row[1] is not None else "<Null>"
            # 還原 Fate 描述
            fate_val = domain_dict.get(row[2], row[2])
            data.append([t_id, r_id, fate_val])
            
    return pd.DataFrame(data, columns=["Temp_ID", "ID", "Fate"])

def print_formatted_result(title, result_df):
    """統一格式輸出函數"""
    print(f"\n{title}")
    if result_df.empty:
        print("   => 檢查通過：未發現衝突。")
    else:
        print("-" * 80)
        header = f"{'Temp_ID':<20} | {'ID':<20} | {'Fate':<12} | {'數量'}"
        print(header)
        print("-" * 80)
        
        # 強制轉換 Temp_ID 為字串型態再排序，解決 TypeError
        result_df = result_df.copy()
        result_df["Temp_ID"] = result_df["Temp_ID"].astype(str)
        result_df = result_df.sort_values(by="Temp_ID")
        
        # 使用 groupby 統計
        for t_id, group in result_df.groupby("Temp_ID"):
            count = len(group)
            # 重設 group 索引以便判斷哪一行是第一筆
            group = group.reset_index(drop=True)
            for i, row in group.iterrows():
                # 只在該 ID 的第一筆紀錄顯示數量
                display_count = count if i == 0 else ""
                print(f"{str(row['Temp_ID']):<20} | {str(row['ID']):<20} | {str(row['Fate']):<12} | {display_count}")
        print("-" * 80)

# --- 執行邏輯 ---
print("正在讀取資料並解析 Domain 代號...")

df_a = get_detailed_df(group_a_codes)
df_b = get_detailed_df(group_b_codes)

# 1. 內部重複 (Fate A 群組內)
internal_dup_ids = df_a[df_a.duplicated(subset="Temp_ID", keep=False)]["Temp_ID"].unique()
df_internal_conflicts = df_a[df_a["Temp_ID"].isin(internal_dup_ids)]

# 2. 交叉衝突 (A 與 B 群組之間)
# 確保比對時型態一致 (皆為字串)
set_a = set(df_a["Temp_ID"].astype(str))
set_b = set(df_b["Temp_ID"].astype(str))
conflict_ids = set_a.intersection(set_b)
conflict_ids.discard("<Null>")

df_cross_conflicts = pd.concat([
    df_a[df_a["Temp_ID"].astype(str).isin(conflict_ids)],
    df_b[df_b["Temp_ID"].astype(str).isin(conflict_ids)]
])

# --- 輸出結果 ---
print_formatted_result(f"【結果一】Fate {group_a_desc} 內部重複檢查", df_internal_conflicts)
print_formatted_result(f"【結果二】Fate {group_b_desc} 與 {group_a_desc} 交叉衝突檢查", df_cross_conflicts)

print("\n檢查結束。")

正在讀取資料並解析 Domain 代號...

【結果一】Fate ['G', 'S', 'GorS'] 內部重複檢查
   => 檢查通過：未發現衝突。

【結果二】Fate ['Fi', 'Fu'] 與 ['G', 'S', 'GorS'] 交叉衝突檢查
   => 檢查通過：未發現衝突。

檢查結束。


In [42]:
##透過 Temp_ID 比對兩期圖層，找出 Form 屬性不一致的記錄，並篩選出非死亡卻在後期資料中遺失的異常樣本

import arcpy

# 1. 設定輸入圖層名稱
t1_layer = "OL_P1_2204_coral_Simplify"
t2_layer = "OL_P1_2209_cor_SimplifyPolyg"

# 2. 定義欄位名稱
t1_fields = ['Temp_ID', 'ID', 'Fate', 'Form']
t2_fields = ['Temp_ID', 'Form']

# --- 自動取得 Domain 對照表 (處理 Fate 欄位) ---
def get_domain_map(feature, field_name):
    desc = arcpy.Describe(feature)
    workspace = desc.path
    domains = arcpy.da.ListDomains(workspace)
    field_domain_name = next((f.domain for f in desc.fields if f.name == field_name), "")
    if field_domain_name:
        for d in domains:
            if d.name == field_domain_name:
                return d.codedValues # {代碼: 描述}
    return {}

# 取得 T1 的 Fate Domain 對照表
t1_fate_domain = get_domain_map(t1_layer, "Fate")

# 3. 建立 T2 的資料字典 (Dictionary)
t2_lookup = {}
print(f"正在讀取 {t2_layer} 的比對資料...")
with arcpy.da.SearchCursor(t2_layer, t2_fields) as cursor:
    for row in cursor:
        t_id, form = row
        if t_id is not None:
            t2_lookup[str(t_id).strip()] = form

# 4. 遍歷 T1 並進行比較
print(f"正在分析 {t1_layer} 與 {t2_layer}...")
print("-" * 80)
print(f"{'Temp_ID':<15} | {'ID':<20} | {'Fate':<10} | {'原因'}")
print("-" * 80)

mismatch_count = 0
missing_non_d_count = 0

with arcpy.da.SearchCursor(t1_layer, t1_fields) as cursor:
    for row in cursor:
        t1_temp_id = str(row[0]).strip() if row[0] is not None else "<Null>"
        t1_id      = row[1]
        t1_fate_raw = row[2]
        t1_form    = row[3]
        
        # 轉換 Fate 代碼為描述 (例如 1 -> 'G', 4 -> 'D')
        t1_fate_desc = t1_fate_domain.get(t1_fate_raw, str(t1_fate_raw))

        # 檢查 T1 的 Temp_ID 是否存在於 T2
        if t1_temp_id in t2_lookup:
            t2_form = t2_lookup[t1_temp_id]
            # 比較 Form 是否不同
            if t1_form != t2_form:
                mismatch_count += 1
                print(f"{t1_temp_id:<15} | {str(t1_id):<20} | {t1_fate_desc:<10} | Form 不符 (T1:{t1_form} vs T2:{t2_form})")
        else:
            # 如果 T2 找不到，且 Fate 不是 'D'，則回報
            if t1_fate_desc != 'D':
                missing_non_d_count += 1
                print(f"{t1_temp_id:<15} | {str(t1_id):<20} | {t1_fate_desc:<10} | 在 T2 中遺失 (非死亡珊瑚)")

# 5. 總結報告
print("-" * 80)
print(f"比對完成報告：")
print(f"1. Form 欄位不符：共 {mismatch_count} 筆。")
print(f"2. 異常遺失 (非 D 但在 T2 找不到)：共 {missing_non_d_count} 筆。")
if missing_non_d_count == 0 and mismatch_count == 0:
    print("結果：資料結構完整，未發現異常。")

正在讀取 OL_P1_2209_cor_SimplifyPolyg 的比對資料...
正在分析 OL_P1_2204_coral_Simplify 與 OL_P1_2209_cor_SimplifyPolyg...
--------------------------------------------------------------------------------
Temp_ID         | ID                   | Fate       | 原因
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
比對完成報告：
1. Form 欄位不符：共 0 筆。
2. 異常遺失 (非 D 但在 T2 找不到)：共 0 筆。
結果：資料結構完整，未發現異常。


In [ ]:
## 重新判定GorS 並顯示Fate比例